In [102]:
import numpy as np
import pandas as pd

db = pd.read_csv("train_transaction.csv")

identity = pd.read_csv("train_identity.csv")

print("Transaction:", db.shape)
print("Identity:", identity.shape)

Transaction: (590540, 394)
Identity: (144233, 41)


In [119]:
data = db.merge(
    identity,
    on="TransactionID",
    how="left"
)

print(data.shape)

(590540, 434)


In [120]:
data = data.sort_values(
    "TransactionDT"
).reset_index(drop=True)

In [121]:
data["isFraud"].value_counts()

,count
isFraud,
0,569877
1,20663


In [122]:
data["has_identity"] = (
    data["id_01"].notna()
).astype(int)

In [123]:
data["has_identity"].value_counts()

,count
has_identity,
0,446307
1,144233


In [124]:
data.groupby("has_identity")["isFraud"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
has_identity,,,
0,446307,9345,0.020939
1,144233,11318,0.078470


In [125]:
transaction_ids = data["TransactionID"].copy()

In [126]:
X = data.drop(
    columns=["isFraud", "TransactionID"]
)

y = data["isFraud"]

print("X:", X.shape)
print("y:", y.shape)

X: (590540, 433)
y: (590540,)


In [129]:
split_1 = int(len(X) * 0.70)
split_2 = int(len(X) * 0.85)

X_train = X.iloc[:split_1].copy()
X_val = X.iloc[split_1:split_2].copy()
X_test = X.iloc[split_2:].copy()

y_train = y.iloc[:split_1].copy()
y_val = y.iloc[split_1:split_2].copy()
y_test = y.iloc[split_2:].copy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (413378, 433)
Validation: (88581, 433)
Test: (88581, 433)


In [130]:
print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_val.mean())
print("Test fraud rate:", y_test.mean())

print("\nFraud counts:")
print("Train:", y_train.sum())
print("Validation:", y_val.sum())
print("Test:", y_test.sum())

Train fraud rate: 0.03516878014795175
Validation fraud rate: 0.03434145019812375
Test fraud rate: 0.03480430340592226

Fraud counts:
Train: 14538
Validation: 3042
Test: 3083


In [131]:
sparse_columns = [
    col for col in X_train.columns
    if X_train[col].isnull().mean() > 0.90
]

print("Sparse columns:", len(sparse_columns))
print(sparse_columns)

Sparse columns: 12
['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']


In [132]:
X_train = X_train.drop(columns=sparse_columns)
X_val = X_val.drop(columns=sparse_columns)
X_test = X_test.drop(columns=sparse_columns)

In [133]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical:", len(categorical_columns))
print("Numerical:", len(numerical_columns))

Categorical: 29
Numerical: 392


In [135]:
train_medians = X_train[numerical_columns].median()

X_train[numerical_columns] = X_train[numerical_columns].fillna(
    train_medians
)

X_val[numerical_columns] = X_val[numerical_columns].fillna(
    train_medians
)

X_test[numerical_columns] = X_test[numerical_columns].fillna(
    train_medians
)

In [136]:
X_train[categorical_columns] = X_train[categorical_columns].fillna(
    "Unknown"
)

X_val[categorical_columns] = X_val[categorical_columns].fillna(
    "Unknown"
)

X_test[categorical_columns] = X_test[categorical_columns].fillna(
    "Unknown"
)

In [137]:
high_cardinality_columns = [
    col for col in categorical_columns
    if X_train[col].nunique() > 100
]

print("High-cardinality columns:")
print(high_cardinality_columns)

High-cardinality columns:
['id_31', 'id_33', 'DeviceInfo']


In [138]:
X_train = X_train.drop(
    columns=high_cardinality_columns
)

X_val = X_val.drop(
    columns=high_cardinality_columns
)

X_test = X_test.drop(
    columns=high_cardinality_columns
)

In [139]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    dummy_na=False
)

X_val = pd.get_dummies(
    X_val,
    columns=categorical_columns,
    dummy_na=False
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    dummy_na=False
)

In [140]:
X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [141]:
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

In [142]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nMissing values:")
print("Train:", X_train.isnull().sum().sum())
print("Validation:", X_val.isnull().sum().sum())
print("Test:", X_test.isnull().sum().sum())

Train: (413378, 663)
Validation: (88581, 663)
Test: (88581, 663)

Missing values:
Train: 0
Validation: 0
Test: 0


In [143]:
import os

os.makedirs("../data/processed", exist_ok=True)

X_train.to_parquet("../data/processed/X_train.parquet")
X_val.to_parquet("../data/processed/X_val.parquet")
X_test.to_parquet("../data/processed/X_test.parquet")

y_train.to_frame().to_parquet("../data/processed/y_train.parquet")
y_val.to_frame().to_parquet("../data/processed/y_val.parquet")
y_test.to_frame().to_parquet("../data/processed/y_test.parquet")

print("Processed data saved successfully.")

Processed data saved successfully.
